In [ ]:
import time
import numpy as np
import tensorflow.compat.v1 as tf
import scipy.sparse as sp
from torch_geometric.data import HeteroData
from models import GAT
from utils import process
import os
from collections import defaultdict

# Disable TF v2 behavior
tf.disable_v2_behavior()

# Configure GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
config = tf.ConfigProto()
config.gpu_options.allow_growth = True

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.set_random_seed(RANDOM_SEED)

def to_numpy(x):
    """Convert tensor to numpy array"""
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.cpu().numpy()
    except ImportError:
        pass
    return np.array(x)

def build_bipartite_adj(src_idx, dst_idx, shape_src, shape_dst):
    """Build a scipy sparse COO adjacency from src to dst"""
    data = np.ones(len(src_idx), dtype=np.float32)
    return sp.coo_matrix((data, (src_idx, dst_idx)), shape=(shape_src, shape_dst))

def build_meta_path_adj(data: HeteroData, meta_path: list):
    """Build adjacency among target nodes via sequence of relations in meta_path"""
    # First relation
    s0, r0, d0 = meta_path[0]
    e0 = to_numpy(data[s0, r0, d0].edge_index)
    A = build_bipartite_adj(e0[0], e0[1], data[s0].num_nodes, data[d0].num_nodes)
    curr = d0
    
    # Remaining relations
    for (src, rel, dst) in meta_path[1:]:
        assert curr == src, f"Meta-path mismatch: expected {curr}, got {src}"
        e = to_numpy(data[src, rel, dst].edge_index)
        B = build_bipartite_adj(e[0], e[1], data[src].num_nodes, data[dst].num_nodes)
        A = A.dot(B)
        curr = dst
    
    A = A.tocoo()
    if A.shape[0] == A.shape[1]:
        A.setdiag(0)
    A.eliminate_zeros()
    return A

def generate_meta_paths(data: HeteroData, max_length=3, start_end='case'):
    """Generate all meta-paths up to max_length hops that start and end at start_end node type"""
    edge_types = data.edge_types
    outgoing = defaultdict(list)
    
    for src, rel, dst in edge_types:
        outgoing[src].append((src, rel, dst))

    results = []
    
    def dfs(path, curr_type):
        if len(path) == max_length:
            if path[0][0] == path[-1][2] == start_end:
                results.append(path.copy())
            return
        
        for edge in outgoing.get(curr_type, []):
            path.append(edge)
            dfs(path, edge[2])
            path.pop()

    for edge in outgoing.get(start_end, []):
        dfs([edge], edge[2])
    
    return results

def prepare_data_from_hetero(data, target_node_type='case', meta_paths=None,
                           split_ratios=(0.7, 0.15, 0.15), random_state=42):
    """Prepare data from heterogeneous graph for GAT training"""
    
    # Features
    if not hasattr(data[target_node_type], 'x'):
        raise ValueError(f"Target node type '{target_node_type}' has no .x features")
    feat = to_numpy(data[target_node_type].x)
    N, ft_size = feat.shape
    
    # Labels
    if not hasattr(data[target_node_type], 'y'):
        raise ValueError(f"Target node type '{target_node_type}' has no .y labels")
    labels = to_numpy(data[target_node_type].y).astype(int).reshape(-1)
    num_classes = labels.max() + 1
    y_all = np.eye(num_classes, dtype=np.float32)[labels]
    
    # Masks
    has_train = hasattr(data[target_node_type], 'train_mask')
    has_val = hasattr(data[target_node_type], 'val_mask')
    has_test = hasattr(data[target_node_type], 'test_mask')
    
    if has_train and has_val and has_test:
        train_mask = to_numpy(data[target_node_type].train_mask).astype(bool)
        val_mask = to_numpy(data[target_node_type].val_mask).astype(bool)
        test_mask = to_numpy(data[target_node_type].test_mask).astype(bool)
    else:
        from sklearn.model_selection import StratifiedShuffleSplit
        idx = np.arange(N)
        s1 = StratifiedShuffleSplit(n_splits=1, test_size=1 - split_ratios[0], random_state=random_state)
        tr_idx, rest = next(s1.split(idx, labels))
        val_prop = split_ratios[1] / (split_ratios[1] + split_ratios[2])
        s2 = StratifiedShuffleSplit(n_splits=1, test_size=1 - val_prop, random_state=random_state)
        v_idx, te_idx_rel = next(s2.split(rest, labels[rest]))
        te_idx = rest[te_idx_rel]
        
        train_mask = np.zeros(N, bool)
        train_mask[tr_idx] = True
        val_mask = np.zeros(N, bool)
        val_mask[rest[v_idx]] = True
        test_mask = np.zeros(N, bool)
        test_mask[te_idx] = True
    
    # Build adjacency matrices for meta-paths
    if meta_paths is None:
        raise ValueError("Provide meta_paths list")
    
    fea_list, bias_list = [], []
    for mp in meta_paths:
        A = build_meta_path_adj(data, mp)
        M = A.toarray()  # shape [N, N]
        fea_list.append(feat[np.newaxis])
        bias = process.adj_to_bias(M[np.newaxis], [N], nhood=1)
        if bias.ndim == 2:
            bias = bias[np.newaxis]
        bias_list.append(bias)
    
    # Labels & masks batched
    y_tr = np.zeros_like(y_all)
    y_tr[train_mask] = y_all[train_mask]
    y_va = np.zeros_like(y_all)
    y_va[val_mask] = y_all[val_mask]
    y_te = np.zeros_like(y_all)
    y_te[test_mask] = y_all[test_mask]
    
    y_tr = y_tr[np.newaxis]
    y_va = y_va[np.newaxis]
    y_te = y_te[np.newaxis]
    
    tm = train_mask[np.newaxis].astype(np.int32)
    vm = val_mask[np.newaxis].astype(np.int32)
    tm_te = test_mask[np.newaxis].astype(np.int32)
    
    return (fea_list, bias_list, y_tr, y_va, y_te,
            tm, vm, tm_te, N, ft_size, num_classes, y_all)

def run_gat_for_path(data: HeteroData, meta_path, verbose=True):
    """Run GAT for a single meta-path"""
    
    # Reset graph for clean training
    tf.reset_default_graph()
    
    # Prepare data
    (fea_list, biases_list, y_tr, y_va, y_te,
     m_tr, m_va, m_te, N, ft_size, num_classes, _y_all) = \
        prepare_data_from_hetero(data, 'case', [meta_path])
    
    # Placeholders
    ftr_in = tf.placeholder(tf.float32, (1, N, ft_size), 'ftr')
    bias_in = tf.placeholder(tf.float32, (1, N, N), 'bias')
    lbl_in = tf.placeholder(tf.int32, (1, N, num_classes), 'lbl')
    msk_in = tf.placeholder(tf.int32, (1, N), 'msk')
    attn_drop = tf.placeholder(tf.float32, (), 'adp')
    ffd_drop = tf.placeholder(tf.float32, (), 'fdp')
    is_train = tf.placeholder(tf.bool, (), 'trn')
    
    # Model
    logits = GAT.inference(
        inputs=ftr_in,
        nb_classes=num_classes,
        nb_nodes=N,
        training=is_train,
        attn_drop=attn_drop,
        ffd_drop=ffd_drop,
        bias_mat=bias_in,
        hid_units=[8],
        n_heads=[8, 1],
        activation=tf.nn.elu,
        residual=False
    )
    
    log_r = tf.reshape(logits, [-1, num_classes])
    lab_r = tf.reshape(lbl_in, [-1, num_classes])
    m_r = tf.reshape(msk_in, [-1])
    
    loss = GAT.masked_softmax_cross_entropy(log_r, lab_r, m_r)
    acc = GAT.masked_accuracy(log_r, lab_r, m_r)
    train_op = GAT.training(loss, 0.005, 0.001)
    
    saver = tf.train.Saver()
    init = tf.group(tf.global_variables_initializer(), tf.local_variables_initializer())
    
    with tf.Session(config=config) as sess:
        sess.run(init)
        best_va_acc = 0.0
        best_va_loss = np.inf
        wait = 0
        
        for ep in range(200):
            # Training
            sess.run(train_op, {
                ftr_in: fea_list[0], bias_in: biases_list[0],
                lbl_in: y_tr, msk_in: m_tr,
                is_train: True, attn_drop: 0.6, ffd_drop: 0.6
            })
            
            # Validation
            l_va, a_va = sess.run([loss, acc], {
                ftr_in: fea_list[0], bias_in: biases_list[0],
                lbl_in: y_va, msk_in: m_va,
                is_train: False, attn_drop: 0.0, ffd_drop: 0.0
            })
            
            if a_va > best_va_acc or (a_va == best_va_acc and l_va < best_va_loss):
                best_va_acc = a_va
                best_va_loss = l_va
                saver.save(sess, 'gat_best.ckpt')
                wait = 0
            else:
                wait += 1
                if wait >= 50:
                    break
        
        # Test
        saver.restore(sess, 'gat_best.ckpt')
        l_te, a_te = sess.run([loss, acc], {
            ftr_in: fea_list[0], bias_in: biases_list[0],
            lbl_in: y_te, msk_in: m_te,
            is_train: False, attn_drop: 0.0, ffd_drop: 0.0
        })
        
        return float(a_te)

def run_gat_experiment(hetero_data: HeteroData, max_path_len=3, top_k_paths=10):
    """Run GAT experiment on multiple meta-paths"""
    
    print("="*80)
    print("GAT CANCER SUBTYPE CLASSIFICATION EXPERIMENT")
    print("="*80)
    
    # Generate meta-paths
    all_paths = generate_meta_paths(hetero_data, max_path_len)
    print(f"→ Found {len(all_paths)} meta-paths of length <= {max_path_len}")
    
    if len(all_paths) == 0:
        print("No valid meta-paths found!")
        return None
    
    results = []
    
    # Test each meta-path
    for i, mp in enumerate(all_paths, 1):
        path_str = " -> ".join([f"{s}--{r}-->{d}" for s, r, d in mp])
        print(f"\n[{i}/{len(all_paths)}] Testing path: {path_str}")
        
        try:
            start_time = time.time()
            acc = run_gat_for_path(hetero_data, mp)
            training_time = time.time() - start_time
            
            print(f"  ✓ Test accuracy: {acc:.4f} (trained in {training_time:.2f}s)")
            results.append((mp, acc, path_str))
            
        except Exception as e:
            print(f"  ✗ Skipped: {e}")
    
    if not results:
        print("No successful experiments!")
        return None
    
    # Sort by accuracy
    results.sort(key=lambda x: -x[1])
    
    print("\n" + "="*80)
    print("GAT EXPERIMENT RESULTS")
    print("="*80)

    print(f"\nTop {min(top_k_paths, len(results))} Meta-paths:")
    for i, (mp, acc, path_str) in enumerate(results[:top_k_paths], 1):
        print(f"{i:2d}. Accuracy: {acc:.4f} | Path: {path_str}")
    
    best_mp, best_acc, best_path_str = results[0]
    print(f"\n🏆 Best GAT result:")
    print(f"   Accuracy: {best_acc:.4f}")
    print(f"   Meta-path: {best_path_str}")
    
    return {
        'best_metapath': best_mp,
        'best_accuracy': best_acc,
        'best_path_string': best_path_str,
        'all_results': results,
        'model_type': 'GAT'
    }

def main(hetero_data, max_path_length=3, top_k=10):
    """Main function to run GAT experiment"""
    return run_gat_experiment(hetero_data, max_path_length, top_k)
